# 14_ZERO-SHOT-LLM_SELF-EXPLAIN

**Role.** Zero-shot instruction-tuned LLM, outside the controlled comparison (`in_comparison: False`). An open-weight model (`Qwen/Qwen3-4B-Instruct-2507`, Apache-2.0, pinned revision; preregistered fallback `microsoft/Phi-4-mini-instruct`, MIT) is asked whether each post is extremist. P(EXTREMIST) comes from the log-probabilities of its two answer tokens, normalised over the two. Nothing is trained on this dataset. The model is then asked which words it relied on, and those self-explanations are compared with its own SHAP attributions.

**Preregistration.** `docs/preregistrations/WP10_zero_shot_llm.md` fixes the model, revision, prompts, label tokens, explainer settings, the self-explanation tests, their Holm family and the reading rule. Nothing here departs from it.

**Research questions.** RQ1 outside the comparison: its own Holm family against every comparison technique (`tools/audit.py`). RQ2 to RQ4 through the two-artifact contract (probability artifact and two whole-model SHAP runs). C2 and C3: are the model's self-explanations faithful to the model?

**Data handling.** The model runs on the kernel's GPU. No dataset text is sent to a hosted API: the kernel only downloads the weights.

**Kaggle inputs.**
1. This repository as an input dataset (provides `tools/notebook_kit.py` and the committed split mirror).
2. The output of `00_create_dataset_and_splits` (the `research_foundation/` folder).
3. Internet on, to download the pinned checkpoint from the Hugging Face Hub (no token needed; neither model is gated). With internet off, attach the same checkpoint at the same revision as a Kaggle model and set `local_model_path` in `CONFIG` to its folder.

**Accelerator.** GPU T4 (one of the two T4s is used), in float16. Do not use the P100: recent PyTorch CUDA builds no longer support Pascal GPUs.

**Software.** `transformers` 4.51 or later. If section 4 stops on the version, add `%pip install -U "transformers>=4.51"` as the first line of the bootstrap cell, restart the kernel and run all.

**Runtime.** The two SHAP runs score about 56,000 masked strings per split (200 posts at `max_evals` 500; short posts use fewer evaluations), roughly 1.5 to 3.5 hours for both on one T4. Scoring 900 posts, the padding check, the self-explanations and the deletion test take minutes. This fits one 12-hour Kaggle session.

**If the model is refused.** Section 4 or 5 stops with a reason when the preregistered fallback rule applies (the checkpoint does not load, a label token is not a single token, a validation logit is non-finite, or the validation format check fails). Then set `"model_choice": "fallback"` and `"fallback_reason"` to that reason (`checkpoint_load`, `label_token`, `non_finite_logits` or `format_check`), and run all again. Never switch because of accuracy. The fallback window closes when the section 5 checks pass: a later failure is not a fallback condition, because switching would evaluate a second model's test split. Such a run is reported, with its partial outputs, under a new version of the preregistration.

**Outputs.**
- `results_summary/14_ZERO-SHOT-LLM_SELF-EXPLAIN/` and `research_loop/probs/14_ZERO-SHOT-LLM_SELF-EXPLAIN__*`: text-free, copied into the repository. Besides the standard files, `self_explanation_test.json` and `self_explanation_words_test.csv` (word counts across posts; words named in fewer than two posts are dropped).
- `external/14_ZERO-SHOT-LLM_SELF-EXPLAIN/`: per-post predictions, per-post SHAP token tables and the per-post self-explanations (`self_explanations/`, which can echo post text). Uploaded to a Kaggle Dataset and recorded in the run manifest with `--contains-text`; never committed. No weights are saved: the checkpoint is on the Hub at the pinned revision.

**Protocol.** There is one preregistered configuration and no training. The decision threshold is selected on the validation split. The test split is evaluated once, in section 8: a new technique, so the test budget allows this one evaluation. Every attribution comes from one method, SHAP's partition explainer over a word-level text masker, so techniques are comparable.

**What this notebook never shows.** Cell outputs are committed, so no cell prints, displays or plots a post, a row id, a text hash or a model reply. Outputs are counts, metrics, configurations and word-level tables.

**Responsible use.** A research artifact for studying what classifiers of extremist content rely on. Labels are post-level and features are text only; nothing here supports decisions about people, accounts or communities.

## 1. Bootstrap

Locate and load the shared notebook kit (`tools/notebook_kit.py`), then import the libraries this notebook uses. The first block is identical in every notebook.

In [ ]:
import importlib.util
import sys
from pathlib import Path

_kit_candidates = (
    sorted(Path("/kaggle/input").rglob("notebook_kit.py"))
    if Path("/kaggle/input").exists()
    else []
)
_kit_candidates += [p / "tools" / "notebook_kit.py" for p in (Path.cwd(), *Path.cwd().parents)]
_kit_path = next((p for p in _kit_candidates if p.is_file()), None)
if _kit_path is None:
    raise FileNotFoundError(
        "notebook_kit.py not found. Attach the repository (or its tools/ folder) "
        "to this kernel as an input dataset."
    )
_kit_spec = importlib.util.spec_from_file_location("notebook_kit", _kit_path)
nk = importlib.util.module_from_spec(_kit_spec)
sys.modules["notebook_kit"] = nk
_kit_spec.loader.exec_module(nk)

import hashlib
import json
import re
import time

import numpy as np
import pandas as pd
import torch
import transformers
from scipy.stats import wilcoxon
from transformers import AutoModelForCausalLM, AutoTokenizer

pd.set_option("display.max_columns", 50)

## 2. Configuration

Everything that defines the run is in `CONFIG`, copied from the preregistration: the model and its pinned revision, the two prompts, the label tokens, the explainer settings and the self-explanation settings. `nk.bootstrap` refuses a configuration whose technique name is not the filename stem, that names a different split or dataset version, or that switches text previews on. It seeds every random number generator and creates the output folders.

In [ ]:
CONFIG = {
    # Identity
    "project_name": "extremism_text_classification_comparison",
    "technique_name": "14_ZERO-SHOT-LLM_SELF-EXPLAIN",
    "role": "transformer",
    "in_comparison": False,
    "model_family": "zero_shot_instruction_llm",
    "feature_family": "llm_label_token_log_probabilities",
    "dataset_version": "extremism_dataset_clean_v1",
    "split_version": "split_v1_stratified_70_15_15_seed30",
    "random_seed": 30,
    "positive_label": 1,
    "text_normalization_mode": "identity",
    "preregistration": "docs/preregistrations/WP10_zero_shot_llm.md",
    # The preregistered model and fallback. Switch to "fallback" only under the
    # preregistered rule, and record the reason.
    "model_choice": "primary",
    "fallback_reason": None,
    "models": {
        "primary": {
            "hf_id": "Qwen/Qwen3-4B-Instruct-2507",
            "revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
            "licence": "apache-2.0",
            "min_transformers": "4.51",
        },
        "fallback": {
            "hf_id": "microsoft/Phi-4-mini-instruct",
            "revision": "cfbefacb99257ffa30c83adab238a50856ac3083",
            "licence": "mit",
            "min_transformers": "4.51",
        },
    },
    # Folder of an attached Kaggle model at the same revision, when internet is off.
    "local_model_path": None,
    # Fixed prompts. "{post}" is replaced by the post, verbatim.
    "prompt": {
        "system": (
            "You are a content classifier for a research study on violent extremism "
            "in social-media text."
        ),
        "user_template": (
            "Decide whether the social-media post below is extremist content. A post is "
            "extremist if it promotes, supports, justifies or glorifies violent extremism, "
            "a violent extremist group or ideology, or violence against people because of "
            "who they are. A post that only reports on, quotes, discusses or argues against "
            "extremism is not extremist.\n\n"
            "Post:\n<<<\n{post}\n>>>\n\n"
            "Is this post extremist? Answer with one word: Yes or No."
        ),
        "explanation_request": (
            "Which words in the post most influenced your answer? List at most 5 words, "
            "most influential first, each copied exactly as it appears in the post, "
            "separated by commas. Reply with the list only."
        ),
    },
    # Answer-token variants; each label's score is the log-sum-exp over the
    # variants that are single tokens. "Yes" and "No" must be single tokens.
    "label_tokens": {
        "EXTREMIST": ["Yes", " Yes", "yes", " yes"],
        "NON_EXTREMIST": ["No", " No", "no", " no"],
    },
    "inference": {"batch_size": 32, "min_compute_capability_for_bf16": 8},
    # One preregistered configuration, scored on validation at the screening
    # threshold. Nothing is compared or selected.
    "ablation": {"metric": "accuracy", "screening_threshold": 0.5},
    # Validation only: the model must answer in the label format.
    "format_check": {"min_median_label_mass": 0.5},
    # Validation only, no labels: added safety check (preregistration amendment
    # log, 2026-11-05). Scores at batch size 1 and inside a padded batch must agree.
    "padding_check": {"n_posts": 16, "max_abs_diff": 1e-2},
    "threshold": {"metric": "accuracy"},
    # SHAP partition explainer over a word-level masker (set by the kit), with
    # the settings of 07 and 11 so the runs compare like for like.
    "explainer": {
        "max_evals": 500,
        "batch_size": 64,
        "seed": 30,
        "n_posts_per_split": 200,
        "per_member_runs": False,
        "top_k_display": 50,
    },
    # Self-explanations on the posts of the test SHAP run.
    "self_explanation": {
        "max_words": 5,
        "max_new_tokens": 32,
        "min_support": 2,
        "random_seeds": [30, 31, 32, 33, 34],
        "holm_family_size": 8,
    },
    "export_coefficients": False,
    "external": {"save_predictions": True, "save_self_explanations": True},
    "error_analysis": {"include_text_preview": False},
}
ctx = nk.bootstrap(CONFIG)

## 3. Load foundation and assert the frozen split

Load the processed dataset and the split assignment written by notebook 00. The kit cross-checks the two files against each other and against the manifest, compares the assignment with the committed mirror when the repository is attached, and stops unless the class counts are exactly train 1309/790, validation 281/169, test 280/170 (non-extremist/extremist). The training split is loaded for the check only: the model is applied zero-shot and never sees it.

In [ ]:
frame, foundation = nk.load_foundation(ctx)
train_df, val_df, test_df = nk.split_frames(frame)

val_texts = val_df["text"].tolist()

print("Rows per split:", {"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
print("Input file hashes:", json.dumps(foundation["file_sha256_16"], indent=2))
print("Training rows used by this technique: 0 (zero-shot)")

## 4. Features and model

The model is the pinned instruction-tuned checkpoint as published, loaded once on the GPU. Each post is placed in the fixed classification prompt and passed through the model's own chat template, with the generation prompt appended, so the next token starts the answer. At that position the score of each label is the log-sum-exp of the log-probabilities of its single-token variants, and P(EXTREMIST) is the `Yes` score normalised over the two labels. Their combined probability, the label mass, says how much of the model's answer is in the label format.

`predict_proba_texts` maps raw strings to P(EXTREMIST) this way. It is the single entry point used for the validation threshold, the test evaluation and every explainer run. Batches are left-padded with position ids taken from the attention mask, and only the last position's logits are computed. A non-finite logit stops the run. Until the section 5 checks pass (`FALLBACK_WINDOW_OPEN`), that is preregistered fallback rule (c); after them it is not a fallback condition, and the run is reported under a new version of the preregistration. It is never something to patch.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a GPU. Turn the accelerator on (GPU T4).")
DEVICE = torch.device("cuda")
FALLBACK_REASONS = ("checkpoint_load", "label_token", "non_finite_logits", "format_check")
if CONFIG["model_choice"] == "fallback":
    assert CONFIG["fallback_reason"] in FALLBACK_REASONS, (
        f"fallback_reason must be one of {FALLBACK_REASONS}: the preregistered rules (a) to (d)"
    )
else:
    assert CONFIG["model_choice"] == "primary" and CONFIG["fallback_reason"] is None, (
        "model_choice is 'primary' or 'fallback'; fallback_reason is None for the primary"
    )
MODEL_SPEC = CONFIG["models"][CONFIG["model_choice"]]
# True until the section 5 checks pass. Only inside this window does a failure
# fall under the preregistered fallback rules.
FALLBACK_WINDOW_OPEN = True

installed_version = tuple(int(part) for part in re.findall(r"\d+", transformers.__version__)[:2])
required_version = tuple(int(part) for part in MODEL_SPEC["min_transformers"].split("."))
if installed_version < required_version:
    raise RuntimeError(
        f"transformers {transformers.__version__} is installed; {MODEL_SPEC['hf_id']} needs "
        f"{MODEL_SPEC['min_transformers']} or later. Add `%pip install -U "
        f"\"transformers>={MODEL_SPEC['min_transformers']}\"` to the bootstrap cell, restart "
        "the kernel and run all."
    )

compute_capability = torch.cuda.get_device_capability(0)
DTYPE = (
    torch.bfloat16
    if compute_capability[0] >= CONFIG["inference"]["min_compute_capability_for_bf16"]
    else torch.float16
)
model_source = CONFIG["local_model_path"] or MODEL_SPEC["hf_id"]
model_revision = None if CONFIG["local_model_path"] else MODEL_SPEC["revision"]

load_start = time.time()
# transformers 4.56 renamed `torch_dtype` to `dtype`.
dtype_argument = "dtype" if installed_version >= (4, 56) else "torch_dtype"
try:
    tokenizer = AutoTokenizer.from_pretrained(model_source, revision=model_revision)
    model = AutoModelForCausalLM.from_pretrained(
        model_source, revision=model_revision, device_map={"": 0}, **{dtype_argument: DTYPE}
    )
except Exception as error:
    if CONFIG["model_choice"] != "primary":
        raise
    raise RuntimeError(
        f"{MODEL_SPEC['hf_id']} did not load at the pinned revision ({type(error).__name__}). "
        "First check that internet is on (or that local_model_path points to the attached "
        "checkpoint at the same revision). If the pinned checkpoint itself cannot be loaded, "
        "preregistered fallback rule (a) applies: set model_choice to 'fallback' and "
        "fallback_reason to 'checkpoint_load', then run all."
    ) from error
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
load_seconds = time.time() - load_start


def single_token_ids(variants):
    """{variant: token id} for the variants the tokenizer encodes as exactly one token."""
    found = {}
    for variant in variants:
        encoded = tokenizer.encode(variant, add_special_tokens=False)
        if len(encoded) == 1:
            found[variant] = int(encoded[0])
    return found


LABEL_TOKEN_IDS = {
    label: single_token_ids(variants) for label, variants in CONFIG["label_tokens"].items()
}
for label, canonical in (("EXTREMIST", "Yes"), ("NON_EXTREMIST", "No")):
    if canonical not in LABEL_TOKEN_IDS[label]:
        raise RuntimeError(
            f"'{canonical}' is not a single token for {MODEL_SPEC['hf_id']}. Preregistered "
            "fallback rule (b): set model_choice to 'fallback' and fallback_reason to "
            "'label_token', then run all."
        )
YES_IDS = sorted(set(LABEL_TOKEN_IDS["EXTREMIST"].values()))
NO_IDS = sorted(set(LABEL_TOKEN_IDS["NON_EXTREMIST"].values()))
if set(YES_IDS) & set(NO_IDS):
    raise RuntimeError("a token id is shared by both labels")
YES_INDEX = torch.tensor(YES_IDS, device=DEVICE)
NO_INDEX = torch.tensor(NO_IDS, device=DEVICE)
PROMPT_SHA256 = hashlib.sha256(
    json.dumps(CONFIG["prompt"], sort_keys=True).encode("utf-8")
).hexdigest()


def chat_turn(speaker, content):
    """One chat message in the format every chat template reads."""
    return dict(role=speaker, content=content)


def classification_messages(post):
    """The fixed system message and classification prompt for one post."""
    return [
        chat_turn("system", CONFIG["prompt"]["system"]),
        chat_turn("user", CONFIG["prompt"]["user_template"].replace("{post}", str(post))),
    ]


def render(messages):
    """The model's own chat template, with the generation prompt appended."""
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def encode_batch(prompts):
    """Left-padded tensors on the GPU, with position ids that skip the padding."""
    encoded = tokenizer(
        prompts, return_tensors="pt", padding=True, add_special_tokens=False
    ).to(DEVICE)
    position_ids = (encoded["attention_mask"].cumsum(-1) - 1).clamp(min=0)
    return encoded, position_ids


def score_texts(strings, batch_size=None):
    """(P(EXTREMIST), label mass) per string, from the first answer token."""
    prompts = [render(classification_messages(value)) for value in strings]
    order = np.argsort([len(prompt) for prompt in prompts], kind="stable")
    probabilities = np.zeros(len(prompts))
    label_mass = np.zeros(len(prompts))
    batch_size = batch_size or CONFIG["inference"]["batch_size"]
    with torch.inference_mode():
        for start in range(0, len(order), batch_size):
            batch_index = order[start : start + batch_size]
            encoded, position_ids = encode_batch([prompts[i] for i in batch_index])
            logits = model(
                input_ids=encoded["input_ids"],
                attention_mask=encoded["attention_mask"],
                position_ids=position_ids,
                use_cache=False,
                logits_to_keep=1,
            ).logits[:, -1, :].float()
            if not torch.isfinite(logits).all():
                if FALLBACK_WINDOW_OPEN:
                    raise FloatingPointError(
                        "non-finite logits on validation. Preregistered fallback rule (c): set "
                        "model_choice to 'fallback' and fallback_reason to 'non_finite_logits', "
                        "then run all."
                    )
                raise FloatingPointError(
                    "non-finite logits after the section 5 checks passed. This is NOT a fallback "
                    "condition: switching models now would evaluate a second model's test split, "
                    "which the preregistration forbids. Stop here, keep the partial outputs, and "
                    "report this run under a new version of the preregistration."
                )
            log_probs = torch.log_softmax(logits, dim=-1)
            score_yes = torch.logsumexp(log_probs[:, YES_INDEX], dim=-1)
            score_no = torch.logsumexp(log_probs[:, NO_INDEX], dim=-1)
            probabilities[batch_index] = torch.sigmoid(score_yes - score_no).cpu().numpy()
            label_mass[batch_index] = (score_yes.exp() + score_no.exp()).cpu().numpy()
    return probabilities, label_mass


def predict_proba_texts(texts):
    """P(EXTREMIST) for a list of strings: the one function every evaluation and explainer uses."""
    return score_texts([str(value) for value in texts])[0]


print("Model:", MODEL_SPEC["hf_id"], "revision", model_revision or "(attached folder)")
print("Device:", torch.cuda.get_device_name(0), "compute capability", compute_capability)
print("Precision:", DTYPE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f} B")
print("Label token ids:", json.dumps(LABEL_TOKEN_IDS))
print("Prompt sha256:", PROMPT_SHA256)
print(f"Load time: {load_seconds / 60:.1f} minutes")

## 5. Ablation on validation

There is nothing to compare: the preregistration fixes one model, one prompt and one decoding rule, so the ablation table has one row, scored on validation at the screening threshold. The same pass runs the preregistered format check: every logit finite and a median label mass of at least 0.5 over the validation posts. A failed check stops the run and points to the fallback; it looks at no label.

A padding check, added as a safety check before any run (amendment log of the preregistration), scores the first 16 validation posts one at a time and again inside one left-padded batch, and compares both with the screening scores. The largest difference must stay at or below 0.01, or the run stops, because then the scores would depend on which posts share a batch. It uses no labels and changes no analysis. When both checks pass, the fallback window closes. The test split is not touched.

In [ ]:
screening_threshold = CONFIG["ablation"]["screening_threshold"]
screening_start = time.time()
val_prob, val_label_mass = score_texts(val_texts)
screening_seconds = time.time() - screening_start

format_check = {
    "split": "validation",
    "n_posts": int(len(val_label_mass)),
    "median_label_mass": float(np.median(val_label_mass)),
    "min_label_mass": float(np.min(val_label_mass)),
    "share_label_mass_below_0_5": float(np.mean(val_label_mass < 0.5)),
    "min_median_label_mass": CONFIG["format_check"]["min_median_label_mass"],
}
format_check["passed"] = bool(
    format_check["median_label_mass"] >= format_check["min_median_label_mass"]
)

padding_settings = CONFIG["padding_check"]
padding_posts = val_texts[: padding_settings["n_posts"]]
single_prob = score_texts(padding_posts, batch_size=1)[0]
batched_prob = score_texts(padding_posts, batch_size=len(padding_posts))[0]
format_check["padding_check"] = {
    "n_posts": len(padding_posts),
    "max_abs_diff_single_vs_padded_batch": float(np.max(np.abs(single_prob - batched_prob))),
    "max_abs_diff_single_vs_screening_batches": float(
        np.max(np.abs(single_prob - val_prob[: len(padding_posts)]))
    ),
    "max_abs_diff_allowed": padding_settings["max_abs_diff"],
}
padding_diff = max(
    format_check["padding_check"]["max_abs_diff_single_vs_padded_batch"],
    format_check["padding_check"]["max_abs_diff_single_vs_screening_batches"],
)
format_check["padding_check"]["passed"] = bool(padding_diff <= padding_settings["max_abs_diff"])
print("Format check on validation:", json.dumps(format_check, indent=2))
if not format_check["passed"]:
    raise RuntimeError(
        "The model is not answering in the label format. Preregistered fallback rule (d): set "
        "model_choice to 'fallback' and fallback_reason to 'format_check', then run all."
    )
if not format_check["padding_check"]["passed"]:
    raise RuntimeError(
        f"Left-padded batches change P(EXTREMIST) by {padding_diff:.3g} (allowed "
        f"{padding_settings['max_abs_diff']}): the scores depend on which posts share a batch. "
        "This is not one of the preregistered fallback rules. Do not switch models; stop and "
        "report it, and fix the batching under a new version of the preregistration."
    )
FALLBACK_WINDOW_OPEN = False
print("Section 5 checks passed; the fallback window is closed.")

screening_metrics = nk.evaluate(ctx, val_df, val_prob, screening_threshold, "validation")
configuration = {
    "hf_id": MODEL_SPEC["hf_id"],
    "revision": model_revision,
    "prompt_sha256": PROMPT_SHA256,
    "dtype": str(DTYPE),
}
ablation_results = pd.DataFrame(
    [
        nk.ablation_row(
            ctx,
            "config_001",
            configuration,
            screening_metrics,
            notes="the preregistered zero-shot configuration; nothing is compared",
        )
    ]
)
best_row = nk.select_best_config(ablation_results, metric=CONFIG["ablation"]["metric"])

print(f"Validation scoring time: {screening_seconds / 60:.1f} minutes")
display(ablation_results)

## 6. Final fit

Nothing is fitted: the checkpoint is used as published, at the pinned revision, so no weights are saved. This section records what was run (model, revision, precision, device, library versions, prompt hash and label token ids) in `external/auxiliary/model_record.json`.

In [ ]:
fit_seconds = 0.0
model_record = {
    "hf_id": MODEL_SPEC["hf_id"],
    "revision": model_revision,
    "source": "attached_folder" if CONFIG["local_model_path"] else "huggingface_hub",
    "licence": MODEL_SPEC["licence"],
    "model_choice": CONFIG["model_choice"],
    "fallback_reason": CONFIG["fallback_reason"],
    "dtype": str(DTYPE),
    "device": torch.cuda.get_device_name(0),
    "n_parameters": int(sum(p.numel() for p in model.parameters())),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "prompt_sha256": PROMPT_SHA256,
    "label_token_ids": LABEL_TOKEN_IDS,
    "load_seconds": load_seconds,
    "fit_seconds": fit_seconds,
    "trained_on_this_dataset": False,
}
nk.write_json(ctx.external("auxiliary") / "model_record.json", model_record)
print(json.dumps(model_record, indent=2))

## 7. Threshold on validation

Select the decision threshold on the validation split, from the same grid every technique uses (0.05 to 0.95 in steps of 0.005), maximizing validation accuracy. The validation probabilities are the ones scored in section 5 by the same function. The kit refuses to select a threshold on any other split. The threshold is now locked.

In [ ]:
threshold, threshold_sweep = nk.select_threshold(
    val_df, val_prob, metric=CONFIG["threshold"]["metric"]
)
validation_metrics = nk.evaluate(ctx, val_df, val_prob, threshold, "validation")

print(f"Threshold selected on validation ({CONFIG['threshold']['metric']}): {threshold:.3f}")
print(json.dumps(validation_metrics, indent=2))

## 8. Single test evaluation

Evaluate the test split once, at the locked threshold. This is a new technique, so the test budget allows this one evaluation (`docs/RESEARCH_PLAN.md`, section 5). The kit raises if the test split is evaluated a second time in the same run. The test label mass is a diagnostic and changes nothing. With 450 test posts, two techniques must differ by about 14 posts before McNemar's test can tell them apart, so this accuracy is a measurement, not a ranking.

In [ ]:
test_prob, test_label_mass = score_texts(test_df["text"].tolist())
test_metrics = nk.evaluate(ctx, test_df, test_prob, threshold, "test")

print(json.dumps(test_metrics, indent=2))
print(
    f"Test label mass (diagnostic): median {np.median(test_label_mass):.3f}, "
    f"share below 0.5 {np.mean(test_label_mass < 0.5):.3f}"
)
display(nk.confusion_matrix_frame(test_metrics))

In [ ]:
diagnostics_path = nk.plot_diagnostics(ctx, test_df, test_prob, test_metrics)

## 9. Probability artifacts

Write the committed probability artifact: one row per validation and test post with exactly `row_id, split, y_true, y_prob`. Every metric in the result folder, the McNemar comparison against the trained techniques, the identity-term false-positive analysis (RQ4) and any later paired test are computed from these files, so a published number can only change when this artifact changes.

In [ ]:
probability_paths = nk.export_probabilities(
    ctx,
    val_df,
    val_prob,
    test_df,
    test_prob,
    meta={
        "model_family": CONFIG["model_family"],
        "feature_family": CONFIG["feature_family"],
        "hyperparameters": {
            **configuration,
            "label_tokens": CONFIG["label_tokens"],
            "probability": "label-token log-probabilities, normalised over the two labels",
        },
        "selected_threshold": threshold,
        "threshold_strategy": nk.THRESHOLD_STRATEGY,
        "threshold_metric": CONFIG["threshold"]["metric"],
    },
)

## 10. Word-level SHAP attribution runs

Explain the model with SHAP's partition explainer over a word-level text masker. The explainer treats the model as a function from text to P(EXTREMIST): it masks words inside the prompt's post slot and measures how the probability moves. The same call explains every technique in the repository, which is what makes their attributions comparable.

Each run is reduced to the committed word-level table: for every word, the mean signed and mean absolute attribution over the posts that contain it, and the number of such posts. Two whole-model runs are exported, one over the test split and one over the validation split. Each explains a label-stratified sample of 200 posts drawn with the run seed, at `max_evals` 500, the settings of 07 and 11, so the runs compare like for like; the sample size is recorded in each sidecar.

The per-post token tables go to `external/` only. The tables shown below are word-level.

In [ ]:
explainer_settings = CONFIG["explainer"]
explainer_seed = explainer_settings["seed"]

shap_start = time.time()
test_sample = nk.sample_for_explanation(
    test_df, explainer_settings["n_posts_per_split"], explainer_seed
)
test_bundle = nk.explain(ctx, predict_proba_texts, test_sample)
test_attributions = nk.export_attribution_run(
    ctx, test_bundle, f"shap-partition_test_seed{explainer_seed}"
)
print(f"Test run time: {(time.time() - shap_start) / 60:.1f} minutes")
display(nk.top_words(test_attributions, explainer_settings["top_k_display"]))

In [ ]:
shap_start = time.time()
validation_sample = nk.sample_for_explanation(
    val_df, explainer_settings["n_posts_per_split"], explainer_seed
)
validation_bundle = nk.explain(ctx, predict_proba_texts, validation_sample)
validation_attributions = nk.export_attribution_run(
    ctx, validation_bundle, f"shap-partition_validation_seed{explainer_seed}"
)
print(f"Validation run time: {(time.time() - shap_start) / 60:.1f} minutes")
display(nk.top_words(validation_attributions, explainer_settings["top_k_display"]))

**Self-explanations.** On the 200 test posts of the test run, the model is asked which words it relied on, in the same conversation as its answer: the classification prompt, an assistant turn with the stated label (`Yes` if the post is flagged at the locked threshold, `No` otherwise), and the fixed explanation request. Decoding is greedy, at most 32 new tokens. The reply is split into words with the masker's pattern and normalised as the word tables are; at most 5 distinct words are kept, and a named word that is not one of the post's own words is counted as `not_in_post`.

Everything below is signed toward the stated label: support for the answer is P(EXTREMIST) for a post stated EXTREMIST and 1 - P(EXTREMIST) otherwise, and SHAP values are multiplied by -1 for posts stated NON_EXTREMIST.

- *Overlap.* For a post with k distinct named words in it, the share of them among SHAP's top k words of that post (overlap@k), against its random expectation k / (words in the post); also the Jaccard index, the mean percentile rank of the named words in SHAP's ranking (0 is the top), and the share of named words whose SHAP value points toward the stated label.
- *Deletion* (comprehensiveness, as `tools/faithfulness.py`). With m the number of tokens of the named words, three deletions of m tokens: the named tokens (`self`), SHAP's m tokens ranked highest toward the stated label (`shap`), and m random tokens (`random`, five seeds, averaged). The drop is the fall in support for the stated label, from the test probability artifact. The original posts are re-scored in the same call and the largest difference from the artifact is reported.

The two populations `stated_extremist` and `stated_non_extremist` carry the tests; `all` is descriptive. The eight Wilcoxon tests (per population: overlap against its expectation, self vs random, shap vs random, shap vs self) form one Holm family of m = 8; a test that cannot be computed stays NaN and does not shrink the family. The `reading` column applies the preregistered reading rule at 0.05. Replies that do not look like a list (no separator, or more than 5 items) are counted, not re-parsed. Per-post replies and word lists go to `external/self_explanations/` only. Committed: the summary JSON and a word table of counts across posts, without the words named in only one post.

In [ ]:
se_settings = CONFIG["self_explanation"]
test_prob_by_row = dict(zip(test_df["row_id"].astype(str), test_prob))
sample_prob = np.array([test_prob_by_row[str(r)] for r in test_bundle.row_ids])
stated_extremist = sample_prob >= threshold
argmax_extremist = sample_prob >= 0.5


def explanation_prompt(post, says_extremist):
    """The classification turn, the stated label as the assistant's answer, then the request."""
    messages = classification_messages(post) + [
        chat_turn("assistant", "Yes" if says_extremist else "No"),
        chat_turn("user", CONFIG["prompt"]["explanation_request"]),
    ]
    return render(messages)


def generate_replies(prompts):
    """Greedy replies, in prompt order."""
    replies = [""] * len(prompts)
    order = np.argsort([len(prompt) for prompt in prompts], kind="stable")
    batch_size = CONFIG["inference"]["batch_size"]
    with torch.inference_mode():
        for start in range(0, len(order), batch_size):
            batch_index = order[start : start + batch_size]
            encoded, _ = encode_batch([prompts[i] for i in batch_index])
            generated = model.generate(
                **encoded,
                do_sample=False,
                max_new_tokens=se_settings["max_new_tokens"],
                pad_token_id=tokenizer.pad_token_id,
                temperature=None,
                top_p=None,
                top_k=None,
            )
            new_tokens = generated[:, encoded["input_ids"].shape[1] :]
            for position, prompt_index in enumerate(batch_index):
                replies[prompt_index] = tokenizer.decode(
                    new_tokens[position], skip_special_tokens=True
                )
    return replies


def parse_named_words(reply):
    """Distinct normalised words of the reply, in order, cut to the preregistered maximum."""
    words = []
    for item in re.split(r"[,;\n]", str(reply)):
        for piece in re.split(nk.MASKER_PATTERN, item):
            word = nk.normalize_word(piece)
            if word and word not in words:
                words.append(word)
    return words[: se_settings["max_words"]]


generation_start = time.time()
sample_posts = test_sample["text"].tolist()
self_replies = generate_replies(
    [explanation_prompt(post, flag) for post, flag in zip(sample_posts, stated_extremist)]
)
named_per_post = [parse_named_words(reply) for reply in self_replies]
print(f"Self-explanations generated for {len(named_per_post)} posts")
print(f"Generation time: {(time.time() - generation_start) / 60:.1f} minutes")
print("Posts whose stated label differs from the argmax label:", int((stated_extremist != argmax_extremist).sum()))

In [ ]:
def delete_tokens(tokens, positions):
    """The post without the tokens at `positions`; an emptied post becomes the mask token."""
    drop = set(int(p) for p in positions)
    kept = "".join(token for i, token in enumerate(tokens) if i not in drop).strip()
    return kept if kept else nk.MASK_TOKEN


def support_for_stated(probability, says_extremist):
    probability = np.asarray(probability, dtype=float)
    return np.where(says_extremist, probability, 1.0 - probability)


post_records = []
for index, (tokens, values) in enumerate(zip(test_bundle.tokens_per_post, test_bundle.values_per_post)):
    sign = 1.0 if stated_extremist[index] else -1.0
    signed = sign * np.asarray(values, dtype=float)
    token_words = [nk.normalize_word(token) for token in tokens]
    word_score, first_position = {}, {}
    for position, (word, value) in enumerate(zip(token_words, signed)):
        if word:
            word_score[word] = word_score.get(word, 0.0) + float(value)
            first_position.setdefault(word, position)
    ranked = sorted(word_score, key=lambda w: (-word_score[w], first_position[w]))
    named = named_per_post[index]
    in_post = [word for word in named if word in word_score]
    reply_items = [item for item in re.split(r"[,;\n]", str(self_replies[index])) if item.strip()]
    record = {
        "stated_extremist": bool(stated_extremist[index]),
        "reply_without_separator": not re.search(r"[,;\n]", str(self_replies[index]).strip()),
        "reply_more_than_max_items": len(reply_items) > se_settings["max_words"],
        "n_named": len(named),
        "n_in_post": len(in_post),
        "n_not_in_post": len(named) - len(in_post),
        "in_post_words": in_post,
        "tokens": tokens,
        "signed": signed,
    }
    if in_post:
        k = len(in_post)
        top_k = set(ranked[:k])
        n_words = len(ranked)
        record.update(
            {
                "k": k,
                "n_words": n_words,
                "top_k_words": ranked[:k],
                "overlap_at_k": len(top_k & set(in_post)) / k,
                "expected_overlap_random": k / n_words,
                "jaccard": len(top_k & set(in_post)) / len(top_k | set(in_post)),
                "mean_percentile_rank": float(
                    np.mean([ranked.index(w) / (n_words - 1) if n_words > 1 else 0.0 for w in in_post])
                ),
                "share_named_toward_label": float(np.mean([word_score[w] > 0 for w in in_post])),
            }
        )
        self_positions = [p for p, w in enumerate(token_words) if w in set(in_post)]
        m = len(self_positions)
        shap_order = np.lexsort((np.arange(len(signed)), -signed))
        record.update(
            {
                "m": m,
                "self_positions": self_positions,
                "shap_positions": sorted(int(p) for p in shap_order[:m]),
            }
        )
    post_records.append(record)

included = [i for i, record in enumerate(post_records) if record["n_in_post"] > 0]
random_positions_by_seed = []
for seed in se_settings["random_seeds"]:
    rng = np.random.default_rng([int(seed), 0])
    random_positions_by_seed.append(
        {
            i: np.sort(rng.choice(len(post_records[i]["tokens"]), size=post_records[i]["m"], replace=False))
            for i in included
        }
    )

deletion_inputs = [sample_posts[i] for i in included]
deletion_inputs += [delete_tokens(post_records[i]["tokens"], post_records[i]["self_positions"]) for i in included]
deletion_inputs += [delete_tokens(post_records[i]["tokens"], post_records[i]["shap_positions"]) for i in included]
for positions in random_positions_by_seed:
    deletion_inputs += [delete_tokens(post_records[i]["tokens"], positions[i]) for i in included]

deletion_start = time.time()
deletion_scores = predict_proba_texts(deletion_inputs)
n_included = len(included)
rescored_original = deletion_scores[:n_included]
p_self = deletion_scores[n_included : 2 * n_included]
p_shap = deletion_scores[2 * n_included : 3 * n_included]
p_random = deletion_scores[3 * n_included :].reshape(len(se_settings["random_seeds"]), n_included)
p_original = sample_prob[included]
verification_max_abs_diff = float(np.max(np.abs(rescored_original - p_original))) if n_included else float("nan")
if verification_max_abs_diff > CONFIG["padding_check"]["max_abs_diff"]:
    print(
        f"WARNING: re-scored originals differ from the probability artifact by up to "
        f"{verification_max_abs_diff:.3g}, above {CONFIG['padding_check']['max_abs_diff']}. "
        "Report this with the deletion results; drops of that size are not interpretable."
    )
print(f"Posts in the deletion test: {n_included} of {len(post_records)}")
print(f"Re-scored originals vs probability artifact, max |diff|: {verification_max_abs_diff:.2e}")
print(f"Deletion scoring time: {(time.time() - deletion_start) / 60:.1f} minutes")

In [ ]:
def mean_or_nan(values):
    values = np.asarray(values, dtype=float)
    return float(values.mean()) if values.size else float("nan")


def paired_wilcoxon(first, second):
    differences = np.asarray(first, dtype=float) - np.asarray(second, dtype=float)
    if differences.size == 0 or np.all(differences == 0):
        return float("nan")
    return float(wilcoxon(first, second).pvalue)


HOLM_M = se_settings["holm_family_size"]
ALPHA = 0.05


def holm(p_values, m=HOLM_M):
    """Holm step-down over a family of m tests (tools/mcnemar_pairs.holm_adjust).

    The family size stays m: a NaN p-value (a test that could not be computed)
    takes no rank and stays NaN, and does not make the other tests' correction
    milder.
    """
    p_values = np.asarray(p_values, dtype=float)
    adjusted = np.full(len(p_values), np.nan)
    finite = np.flatnonzero(np.isfinite(p_values))
    running = 0.0
    for rank, index in enumerate(finite[np.argsort(p_values[finite], kind="stable")]):
        running = max(running, min(1.0, (m - rank) * p_values[index]))
        adjusted[index] = running
    return adjusted


stated_included = stated_extremist[included]
support_original = support_for_stated(p_original, stated_included)
drop_self = support_original - support_for_stated(p_self, stated_included)
drop_shap = support_original - support_for_stated(p_shap, stated_included)
drop_random_by_seed = support_original[None, :] - support_for_stated(p_random, stated_included[None, :])
drop_random = drop_random_by_seed.mean(axis=0)
flagged_after = {
    "self": p_self >= threshold,
    "shap": p_shap >= threshold,
    "random": p_random >= threshold,
}

POPULATIONS = {
    "stated_extremist": stated_included,
    "stated_non_extremist": ~stated_included,
    "all": np.ones(n_included, dtype=bool),
}
POPULATION_OF_POST = {
    "stated_extremist": lambda r: r["stated_extremist"],
    "stated_non_extremist": lambda r: not r["stated_extremist"],
    "all": lambda r: True,
}
overlap_rows, deletion_rows = [], []
for population, mask in POPULATIONS.items():
    tested = population != "all"
    chosen = [post_records[i] for i, keep in zip(included, mask) if keep]
    members = [r for r in post_records if POPULATION_OF_POST[population](r)]
    overlap = np.array([r["overlap_at_k"] for r in chosen])
    expected = np.array([r["expected_overlap_random"] for r in chosen])
    overlap_rows.append(
        {
            "population": population,
            "n_posts_population": len(members),
            "n_empty_replies": sum(1 for r in members if r["n_named"] == 0),
            "n_named_words": sum(r["n_named"] for r in members),
            "n_named_words_not_in_post": sum(r["n_not_in_post"] for r in members),
            "n_posts": len(chosen),
            "mean_k": mean_or_nan([r["k"] for r in chosen]),
            "mean_n_words": mean_or_nan([r["n_words"] for r in chosen]),
            "mean_overlap_at_k": mean_or_nan(overlap),
            "mean_expected_overlap_random": mean_or_nan(expected),
            "overlap_minus_expected": mean_or_nan(overlap - expected),
            "mean_jaccard": mean_or_nan([r["jaccard"] for r in chosen]),
            "mean_percentile_rank": mean_or_nan([r["mean_percentile_rank"] for r in chosen]),
            "share_named_toward_label": mean_or_nan([r["share_named_toward_label"] for r in chosen]),
            "wilcoxon_p": paired_wilcoxon(overlap, expected) if tested else float("nan"),
        }
    )
    flips = {}
    for name, after in flagged_after.items():
        stated_part = stated_included[mask]
        if name == "random":
            flipped = after[:, mask] != stated_part[None, :]
        else:
            flipped = after[mask] != stated_part
        flips[name] = mean_or_nan(flipped)
    row = {
        "population": population,
        "n_posts": int(mask.sum()),
        "mean_m": mean_or_nan([r["m"] for r in chosen]),
        "mean_support_original": mean_or_nan(support_original[mask]),
        "mean_drop_self": mean_or_nan(drop_self[mask]),
        "mean_drop_shap": mean_or_nan(drop_shap[mask]),
        "mean_drop_random": mean_or_nan(drop_random[mask]),
        "self_minus_random": mean_or_nan(drop_self[mask] - drop_random[mask]),
        "shap_minus_random": mean_or_nan(drop_shap[mask] - drop_random[mask]),
        "shap_minus_self": mean_or_nan(drop_shap[mask] - drop_self[mask]),
        "median_self_minus_random": float(np.median(drop_self[mask] - drop_random[mask])) if mask.any() else float("nan"),
        "share_self_gt_random": mean_or_nan(drop_self[mask] > drop_random[mask]),
        "random_drop_sd_over_seeds": float(drop_random_by_seed[:, mask].mean(axis=1).std(ddof=1)) if mask.any() else float("nan"),
        "flip_share_self": flips["self"],
        "flip_share_shap": flips["shap"],
        "flip_share_random": flips["random"],
        "n_random_seeds": len(se_settings["random_seeds"]),
    }
    for label, first, second in (
        ("self_vs_random", drop_self, drop_random),
        ("shap_vs_random", drop_shap, drop_random),
        ("shap_vs_self", drop_shap, drop_self),
    ):
        row[f"wilcoxon_p_{label}"] = paired_wilcoxon(first[mask], second[mask]) if tested else float("nan")
    deletion_rows.append(row)

overlap_table = pd.DataFrame(overlap_rows)
deletion_table = pd.DataFrame(deletion_rows)
family = [("overlap", i, "wilcoxon_p") for i in range(len(overlap_table))]
family += [
    ("deletion", i, f"wilcoxon_p_{label}")
    for i in range(len(deletion_table))
    for label in ("self_vs_random", "shap_vs_random", "shap_vs_self")
]
family = [
    entry for entry in family
    if (overlap_table if entry[0] == "overlap" else deletion_table).loc[entry[1], "population"] != "all"
]
if len(family) != se_settings["holm_family_size"]:
    raise RuntimeError(f"the Holm family has {len(family)} tests; the preregistration fixes 8")
adjusted = holm(
    [(overlap_table if t == "overlap" else deletion_table).loc[i, c] for t, i, c in family]
)
for (table_name, row_index, column), value in zip(family, adjusted):
    table = overlap_table if table_name == "overlap" else deletion_table
    table.loc[row_index, column + "_holm"] = value


def significant(p_value):
    return bool(np.isfinite(p_value) and p_value < ALPHA)


def deletion_reading(row):
    """The preregistered reading rule for one stated-label population."""
    if row["population"] == "all":
        return "descriptive, untested"
    faithful = significant(row["wilcoxon_p_self_vs_random_holm"]) and row["self_minus_random"] > 0
    shap_ahead = significant(row["wilcoxon_p_shap_vs_self_holm"]) and row["shap_minus_self"] > 0
    readings = []
    if faithful:
        readings.append("faithful at the deletion level")
    if shap_ahead:
        readings.append("less faithful than SHAP")
    if faithful and not significant(row["wilcoxon_p_shap_vs_self_holm"]):
        readings.append("as faithful as SHAP within resolution")
    return "; ".join(readings) if readings else "not classified (described)"


def overlap_reading(row):
    if row["population"] == "all":
        return "descriptive, untested"
    if significant(row["wilcoxon_p_holm"]) and row["overlap_minus_expected"] > 0:
        return "overlap above its random expectation"
    return "not classified (described)"


deletion_table["reading"] = [deletion_reading(row) for _, row in deletion_table.iterrows()]
overlap_table["reading"] = [overlap_reading(row) for _, row in overlap_table.iterrows()]

display(overlap_table)
display(deletion_table)

In [ ]:
word_counts = {}
for record in post_records:
    if not record["n_in_post"]:
        continue
    for word in record["in_post_words"]:
        entry = word_counts.setdefault(word, {"support": 0, "extremist": 0, "non_extremist": 0, "in_top_k": 0})
        entry["support"] += 1
        entry["extremist" if record["stated_extremist"] else "non_extremist"] += 1
        entry["in_top_k"] += int(word in record["top_k_words"])

word_table = pd.DataFrame(
    [
        {
            "word": word,
            "support": entry["support"],
            "support_stated_extremist": entry["extremist"],
            "support_stated_non_extremist": entry["non_extremist"],
            "share_in_shap_top_k": entry["in_top_k"] / entry["support"],
        }
        for word, entry in word_counts.items()
    ],
    columns=[
        "word",
        "support",
        "support_stated_extremist",
        "support_stated_non_extremist",
        "share_in_shap_top_k",
    ],
)
n_words_named = len(word_table)
valid_word = (
    word_table["word"].str.len().between(1, nk.MAX_WORD_LENGTH)
    & ~word_table["word"].str.contains(r"\s", regex=True)
)
word_table = word_table[valid_word & (word_table["support"] >= se_settings["min_support"])]
word_table = word_table.sort_values(["support", "word"], ascending=[False, True]).reset_index(drop=True)
word_table_path = ctx.results_dir / "self_explanation_words_test.csv"
word_table.to_csv(word_table_path, index=False, float_format="%.6g")
ctx.written.append(word_table_path)

summary = {
    "technique": ctx.technique,
    "run_id": ctx.run_id,
    "split": "test",
    "preregistration": CONFIG["preregistration"],
    "attribution_run": f"shap-partition_test_seed{explainer_seed}",
    "hf_id": MODEL_SPEC["hf_id"],
    "revision": model_revision,
    "prompt_sha256": PROMPT_SHA256,
    "settings": {
        "stated_label": "flagged at the locked validation threshold",
        "threshold": threshold,
        "max_words": se_settings["max_words"],
        "max_new_tokens": se_settings["max_new_tokens"],
        "decoding": "greedy",
        "random_seeds": se_settings["random_seeds"],
        "min_support_in_word_table": se_settings["min_support"],
        "holm_family": "8 Wilcoxon tests: overlap, self_vs_random, shap_vs_random, shap_vs_self per stated-label population",
        "holm_m": HOLM_M,
        "alpha": ALPHA,
    },
    "counts": {
        "n_posts_explained": len(post_records),
        "n_posts_stated_extremist": int(stated_extremist.sum()),
        "n_stated_differs_from_argmax": int((stated_extremist != argmax_extremist).sum()),
        "n_empty_replies": int(sum(1 for r in post_records if r["n_named"] == 0)),
        "n_named_words": int(sum(r["n_named"] for r in post_records)),
        "n_named_words_not_in_post": int(sum(r["n_not_in_post"] for r in post_records)),
        "n_replies_without_separator": int(sum(r["reply_without_separator"] for r in post_records)),
        "n_replies_more_than_max_items": int(sum(r["reply_more_than_max_items"] for r in post_records)),
        "n_posts_with_named_word_in_post": n_included,
        "n_distinct_named_words_in_post": n_words_named,
        "n_words_in_committed_table": int(len(word_table)),
    },
    "verification": {"rescored_original_max_abs_diff": verification_max_abs_diff},
    "overlap": overlap_table.to_dict(orient="records"),
    "deletion": deletion_table.to_dict(orient="records"),
}
if re.search(r"\bex_\d{6}\b", json.dumps(summary, default=str)):
    raise RuntimeError("the self-explanation summary carries a row id; nothing is written")
summary_path = nk.write_json(ctx.results_dir / "self_explanation_test.json", summary)
ctx.written.append(summary_path)

if CONFIG["external"]["save_self_explanations"]:
    pd.DataFrame(
        {
            "row_id": test_bundle.row_ids,
            "stated_extremist": stated_extremist,
            "reply": self_replies,
            "named_words": [" ".join(words) for words in named_per_post],
            "named_words_in_post": [" ".join(r["in_post_words"]) for r in post_records],
        }
    ).to_csv(ctx.external("self_explanations") / "self_explanations_test.csv", index=False)

print(json.dumps(summary["counts"], indent=2))
print(f"Words named in at least {se_settings['min_support']} posts: {len(word_table)} (shown: top 30 by support)")
display(word_table.iloc[:30])

## 11. Result folder

Write `results_summary/<technique>/` from the metrics computed above: the configuration, validation and test metrics, the confusion matrix, the classification report, the one-row ablation table and the threshold sweep. `in_comparison` is False, so `tools/audit.py` gives this technique its own Holm family against every comparison technique and keeps it out of the comparison tables. The kit refuses a folder whose locked threshold differs from the one the metrics were computed at, or whose ablation table has a test column.

In [ ]:
best_config = {
    "model_family": CONFIG["model_family"],
    "feature_family": CONFIG["feature_family"],
    "hyperparameters": {
        **configuration,
        "model_source": model_record["source"],
        "prompt": CONFIG["prompt"],
        "label_tokens": CONFIG["label_tokens"],
        "label_token_ids": LABEL_TOKEN_IDS,
        "inference_batch_size": CONFIG["inference"]["batch_size"],
    },
    "threshold_strategy": nk.THRESHOLD_STRATEGY,
    "threshold_metric": CONFIG["threshold"]["metric"],
    "selected_threshold": threshold,
    "selected_config_id": best_row["config_id"],
    "selection_rule": "one preregistered configuration; nothing is selected",
    "n_validation_candidates": int(len(ablation_results)),
    "validation_uses": ["format_check", "threshold"],
    "format_check": format_check,
    "model_choice": CONFIG["model_choice"],
    "fallback_reason": CONFIG["fallback_reason"],
    "licence": MODEL_SPEC["licence"],
    "preregistration": CONFIG["preregistration"],
    "trained_on_this_dataset": False,
    "external_pretraining": "open-weight instruction-tuned LLM as published; zero-shot",
    "fit_seconds": fit_seconds,
    "device": torch.cuda.get_device_name(0),
}
results_folder = nk.export_results_folder(
    ctx, best_config, validation_metrics, test_metrics, ablation_results, threshold_sweep
)

## 12. External assets

Files that hold dataset text are written under `external/`: the per-post predictions a reviewer reads, the per-post attribution tables and the per-post self-explanations. The table below is all that is shown here: test outcomes counted by confidence.

In [ ]:
if CONFIG["external"]["save_predictions"]:
    nk.export_external_predictions(ctx, val_df, val_prob, test_df, test_prob, threshold)

display(nk.error_counts(test_df, test_prob, threshold))

## 13. Interpretation

This section is written before the run and states how to read the outputs. Numbers are not copied here; they live in the result folder and in the outputs of `tools/audit.py`.

**Reading the outputs.** The probability is the model's preference between its two answer tokens, not a trained classifier's score, so it can be poorly calibrated even when the ranking is useful; the threshold selected on validation absorbs a constant bias. The word tables in section 10 are read like those of 07: a word's effect depends on its context, and the signed mean says whether it pushes toward EXTREMIST. RQ1 for this technique is its McNemar comparison with each trained technique in its own Holm family; a gap of fewer than about 14 test posts is not a detectable difference, and no accuracy gap is a statement about zero-shot models in general.

**Self-explanations.** The preregistered reading rule applies per stated-label population. Self-explanations are *faithful at the deletion level* if self vs random is Holm-significant with a positive mean margin; *less faithful than SHAP* if shap vs self is Holm-significant with a positive mean margin; *as faithful as SHAP within resolution* if self vs random is significant and shap vs self is not. Anything else is described, not classified. Overlap@k is read against its random expectation. Named words that are not in the post, and empty replies, are counted, not repaired.

**Limitations.**
- One prompt, one model, one decoding rule. Another wording could change accuracy and explanations. The label definition in the prompt is the authors', not the annotators'.
- The model was not trained on this dataset; its accuracy is not a statement about architectures, and the comparison with trained techniques is not like for like.
- Self-explanations are generated text from the model they describe. Agreement with SHAP is agreement between two descriptions of one model, and deletion measures whether the named words are needed, not whether they suffice.
- SHAP masks words with `...` and deletion removes them; both perturb the post away from real text.
- The local perturbation tools (WP7, WP7b, `faithfulness.py`) cannot restore this model, so those stages skip it; its SHAP deletion faithfulness is the `shap` vs `random` rows above.
- Safety tuning may lower the label mass on some posts; the format check and the test diagnostic report it.
- In float16, left-padded batches move a probability by about 1e-3 depending on which posts share a batch. The padding check in section 5 bounds this at 0.01, and differences of that size (in re-scored originals, deletion drops or SHAP values) are numerical noise, not model behaviour.
- Labels are post-level and features are text only. The model detects extremist content in a post. It does not detect accounts, intent or radicalization over time.
- Lexicon categories (extremist framing, identity term, slur, topical) are assigned afterwards by `tools/categorize_attributions.py`; no category is assigned in this notebook.

## 14. Package outputs

Inventory the external files with their hashes, check that nothing text-bearing sits in the committed tree, and zip the files that go into the repository. The zip unpacks at the repository root.

**After the run.**
1. Unzip `<technique>_repo_files.zip` at the repository root.
2. Upload `external/<technique>/` to a Kaggle Dataset, then record it: `tools/run_manifest.py init`, then `add-assets-from --file external_assets.json`. The `self_explanations/` files hold model replies that can echo posts: record them with `--contains-text`.
3. Run `tools/audit.py run --technique 14_ZERO-SHOT-LLM_SELF-EXPLAIN` and `tools/audit.py report`, then `tools/validate_results_folder.py --all`.
4. Before committing this notebook with its outputs, run `tools/check_notebook.py --notebook <file>` and `tools/scan_text_leakage.py`.

In [ ]:
nk.finalize(ctx)